In [88]:
import pandas as pd

# Load all the excel files:

In [89]:
sheet1 = pd.read_excel("GBP_DataSource1.xlsx", header=0, sheet_name="Orders")
sheet2 = pd.read_excel("GBP_DataSource1.xlsx", header=0, sheet_name="People")
sheet3 = pd.read_excel("GBP_DataSource1.xlsx", header=0, sheet_name="Returns")

def clean_columns(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(r"[^a-z0-9_]", "_", regex=True)
    return df

sheet1 = clean_columns(sheet1)
sheet2 = clean_columns(sheet2)
sheet3 = clean_columns(sheet3)

In [90]:
sheet1.head()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country_region,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1.0,CA-2020-152156,2020-11-08,2020-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2.0,CA-2020-152156,2020-11-08,2020-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3.0,CA-2020-138688,2020-06-12,2020-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4.0,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5.0,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [91]:
sheet2.head()

,regional_manager,region
0,Sadie Pawthorne,West
1,Chuck Magee,East
2,Roxanne Rodriguez,Central
3,Fred Suzuki,South
4,Roxanne Rodriguez,NaN


In [92]:
sheet3.head()

,returned,order_id
0,Yes,CA-2018-100762
1,Yes,CA-2018-100762
2,Yes,CA-2018-100762
3,Yes,CA-2018-100762
4,Yes,CA-2018-100867


# Dealing with missing row_id in sheet1

In [93]:
last_valid_id = None

for i in range(len(sheet1)):
    if pd.isna(sheet1.loc[i, "row_id"]):
        if last_valid_id is not None:
            sheet1.loc[i, "row_id"] = last_valid_id + 1
            last_valid_id += 1
    else:
        last_valid_id = sheet1.loc[i, "row_id"]

# Dealing with duplicate rows in sheet 1 and 3

In [94]:
#dealing with duplicate data - sheet 1

print(sheet1.shape)
print("Number of dulicates:", sheet1.duplicated().sum())
sheet1[sheet1.duplicated(keep=False)]
sheet1 = sheet1.drop_duplicates()           #keeps the first occurance and deletes the rest
print(sheet1.shape)

(10020, 21)
Number of dulicates: 23
(9997, 21)


In [95]:
#dealing with duplicate data - sheet 3

print(sheet3.shape)
print("Number of dulicates:", sheet3.duplicated().sum())
sheet3[sheet3.duplicated(keep=False)]
sheet3 = sheet3.drop_duplicates()           #keeps the first occurance and deletes the rest
print(sheet3.shape)

(800, 2)
Number of dulicates: 502
(298, 2)


# Dealing with missing country_region in sheet1

In [96]:
sheet1.isnull().sum()

row_id             0
order_id           2
order_date         2
ship_date          1
ship_mode          1
customer_id        0
customer_name      1
segment            2
country_region     2
city               0
state              1
postal_code       11
region             0
product_id         0
category           0
sub_category       0
product_name       0
sales              0
quantity           0
discount           0
profit             0
dtype: int64

In [97]:
#Imputing Country with mode value and changing all values to US 

mode_v = sheet1["country_region"].mode()[0]
mask = sheet1["country_region"].isna() | (sheet1["country_region"] != mode_v)
changed_rows = sheet1.loc[mask, "country_region"].copy()
sheet1.loc[mask, "country_region"] = mode_v

num_changed = mask.sum()
print("Number of rows changed:", num_changed)

report = pd.DataFrame({"index": changed_rows.index,"previous_value": changed_rows.values,"new_value": mode_v})
print(report)


Number of rows changed: 4
   index previous_value      new_value
0   9338            NaN  United States
1   9348            NaN  United States
2  10012         Canada  United States
3  10013         Canada  United States


In [98]:
sheet1.isnull().sum()

row_id             0
order_id           2
order_date         2
ship_date          1
ship_mode          1
customer_id        0
customer_name      1
segment            2
country_region     0
city               0
state              1
postal_code       11
region             0
product_id         0
category           0
sub_category       0
product_name       0
sales              0
quantity           0
discount           0
profit             0
dtype: int64

# Dealing with missing values in sheet 2

In [99]:
sheet2.isnull().sum()

regional_manager    0
region              1
dtype: int64

In [100]:
# Here we decided to delete the row with the duplicate manager name

sheet2 = sheet2.dropna()
sheet2.isnull().sum()


regional_manager    0
region              0
dtype: int64

In [101]:
sheet2

,regional_manager,region
0,Sadie Pawthorne,West
1,Chuck Magee,East
2,Roxanne Rodriguez,Central
3,Fred Suzuki,South
5,George Smith,North


# Splitting first and last name

In [102]:
sheet2[["first_name", "last_name"]] = \
sheet2["regional_manager"].str.split(" ", expand=True)
sheet2 = sheet2.drop(columns="regional_manager")

In [103]:
sheet2

,region,first_name,last_name
0,West,Sadie,Pawthorne
1,East,Chuck,Magee
2,Central,Roxanne,Rodriguez
3,South,Fred,Suzuki
5,North,George,Smith


# Dealing with missing values in sheet 3

In [104]:
sheet3.isnull().sum()

returned    0
order_id    1
dtype: int64

In [105]:
# Here we decided to delete the row with the missing order id, since we wont be able to get the missing order id

sheet3 = sheet3.dropna()
sheet3.isnull().sum()

returned    0
order_id    0
dtype: int64

# Dealing with missing customer_name in sheet1

In [106]:
sheet1["customer_name"] = sheet1.groupby("customer_id")["customer_name"].transform(lambda x: x.fillna(method="ffill").fillna(method="bfill"))

/var/folders/r2/yr0p4gxd1c541cyhkm3mbl5c0000gn/T/ipykernel_62632/3480818656.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  sheet1["customer_name"] = sheet1.groupby("customer_id")["customer_name"].transform(lambda x: x.fillna(method="ffill").fillna(method="bfill"))


In [107]:
sheet1.isnull().sum()

row_id             0
order_id           2
order_date         2
ship_date          1
ship_mode          1
customer_id        0
customer_name      0
segment            2
country_region     0
city               0
state              1
postal_code       11
region             0
product_id         0
category           0
sub_category       0
product_name       0
sales              0
quantity           0
discount           0
profit             0
dtype: int64

# Dealing with missing segment in sheet1

In [108]:
sheet1["segment"] = sheet1.groupby("customer_id")["segment"].transform(lambda x: x.fillna(method="ffill").fillna(method="bfill"))

/var/folders/r2/yr0p4gxd1c541cyhkm3mbl5c0000gn/T/ipykernel_62632/938198156.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  sheet1["segment"] = sheet1.groupby("customer_id")["segment"].transform(lambda x: x.fillna(method="ffill").fillna(method="bfill"))


In [109]:
sheet1.isnull().sum()

row_id             0
order_id           2
order_date         2
ship_date          1
ship_mode          1
customer_id        0
customer_name      0
segment            0
country_region     0
city               0
state              1
postal_code       11
region             0
product_id         0
category           0
sub_category       0
product_name       0
sales              0
quantity           0
discount           0
profit             0
dtype: int64

# Dealing with missing state in sheet1

In [110]:
sheet1["state"] = sheet1.groupby("city")["state"].transform(lambda x: x.fillna(method="ffill").fillna(method="bfill"))

/var/folders/r2/yr0p4gxd1c541cyhkm3mbl5c0000gn/T/ipykernel_62632/746306981.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  sheet1["state"] = sheet1.groupby("city")["state"].transform(lambda x: x.fillna(method="ffill").fillna(method="bfill"))


In [111]:
sheet1.isnull().sum()

row_id             0
order_id           2
order_date         2
ship_date          1
ship_mode          1
customer_id        0
customer_name      0
segment            0
country_region     0
city               0
state              0
postal_code       11
region             0
product_id         0
category           0
sub_category       0
product_name       0
sales              0
quantity           0
discount           0
profit             0
dtype: int64

# Dealing with missing ship_mode in sheet1

In [112]:
index=sheet1[sheet1["ship_mode"].isna()].index.tolist()
order_id = sheet1["order_id"][index].iloc[0]
print(order_id)

value = sheet1.loc[sheet1["order_id"] == order_id, "ship_mode"].iloc[0]

sheet1["ship_mode"] = sheet1['ship_mode'].fillna(value)

CA-2021-140872


In [113]:
sheet1.isnull().sum()

row_id             0
order_id           2
order_date         2
ship_date          1
ship_mode          0
customer_id        0
customer_name      0
segment            0
country_region     0
city               0
state              0
postal_code       11
region             0
product_id         0
category           0
sub_category       0
product_name       0
sales              0
quantity           0
discount           0
profit             0
dtype: int64

# Dealing with missing order_date and ship_date in sheet1

In [114]:
#Converting to datetime
sheet1["order_date"] = pd.to_datetime(sheet1["order_date"])
sheet1["ship_date"] = pd.to_datetime(sheet1["ship_date"])

#Calculating shipping days
sheet1["ship_days"] = (sheet1["ship_date"] - sheet1["order_date"]).dt.days

#Get typical shipping time per ship_mode
ship_mode_days = sheet1.groupby("ship_mode")["ship_days"].transform(lambda x: x.mode()[0] if not x.mode().empty else None)

In [115]:
# Fill missing ship_date (only within same ship_mode)
sheet1.loc[sheet1["ship_date"].isna(), "ship_date"] = (sheet1["order_date"] + pd.to_timedelta(ship_mode_days, unit="D"))

# Fill missing order_date
sheet1.loc[sheet1["order_date"].isna(), "order_date"] = (sheet1["ship_date"] - pd.to_timedelta(ship_mode_days, unit="D"))

sheet1 = sheet1.drop(columns=['ship_days'])

In [116]:
sheet1.isnull().sum()

row_id             0
order_id           2
order_date         0
ship_date          0
ship_mode          0
customer_id        0
customer_name      0
segment            0
country_region     0
city               0
state              0
postal_code       11
region             0
product_id         0
category           0
sub_category       0
product_name       0
sales              0
quantity           0
discount           0
profit             0
dtype: int64

# Dealing with missing order_id in sheet1 - We decided to manually impute these values

In [117]:
sheet1[sheet1["order_id"].isna()]

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country_region,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
8954,8938.0,NaN,2019-12-17,2019-12-19,Second Class,PG-18895,Paul Gonzalez,Consumer,United States,Los Angeles,...,90008.0,West,TEC-PH-10004908,Technology,Phones,Panasonic KX TS3282W Corded phone,135.984,2,0.2,16.998
9217,9201.0,NaN,2020-10-17,2020-10-20,First Class,NR-18550,Nick Radford,Consumer,United States,Perth Amboy,...,8861.0,East,FUR-BO-10001337,Furniture,Bookcases,O'Sullivan Living Dimensions 2-Shelf Bookcases,120.980,1,0.0,12.098


In [118]:
filtered_df = sheet1[(sheet1["customer_name"] == "Paul Gonzalez") & (sheet1["ship_date"] == "2019-12-19") & (sheet1["order_date"] == "2019-12-17")]
filtered_df

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country_region,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
8953,8937.0,CA-2019-134117,2019-12-17,2019-12-19,Second Class,PG-18895,Paul Gonzalez,Consumer,United States,Los Angeles,...,90008.0,West,OFF-AR-10003903,Office Supplies,Art,Sanford 52201 APSCO Electric Pencil Sharpener,204.850,5,0.0,53.261
8954,8938.0,NaN,2019-12-17,2019-12-19,Second Class,PG-18895,Paul Gonzalez,Consumer,United States,Los Angeles,...,90008.0,West,TEC-PH-10004908,Technology,Phones,Panasonic KX TS3282W Corded phone,135.984,2,0.2,16.998
8955,8939.0,CA-2019-134117,2019-12-17,2019-12-19,Second Class,PG-18895,Paul Gonzalez,Consumer,United States,Los Angeles,...,90008.0,West,OFF-AR-10001940,Office Supplies,Art,"Sanford Colorific Eraseable Coloring Pencils, ...",16.400,5,0.0,7.052
8956,8940.0,CA-2019-134117,2019-12-17,2019-12-19,Second Class,PG-18895,Paul Gonzalez,Consumer,United States,Los Angeles,...,90008.0,West,OFF-BI-10002026,Office Supplies,Binders,Avery Arch Ring Binders,92.960,2,0.2,31.374


In [119]:
sheet1.loc[(sheet1["order_id"].isna()) & (sheet1["customer_name"] == "Paul Gonzalez"),"order_id"] = "CA-2019-134117"

In [120]:
filtered_df = sheet1[(sheet1["customer_name"] == "Nick Radford") & (sheet1["ship_date"] == "2020-10-20") & (sheet1["order_date"] == "2020-10-17")]
filtered_df

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country_region,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
9217,9201.0,NaN,2020-10-17,2020-10-20,First Class,NR-18550,Nick Radford,Consumer,United States,Perth Amboy,...,8861.0,East,FUR-BO-10001337,Furniture,Bookcases,O'Sullivan Living Dimensions 2-Shelf Bookcases,120.98,1,0.0,12.0980
9218,9202.0,CA-2020-152688,2020-10-17,2020-10-20,First Class,NR-18550,Nick Radford,Consumer,United States,Perth Amboy,...,8861.0,East,OFF-BI-10004584,Office Supplies,Binders,GBC ProClick 150 Presentation Binding System,315.98,1,0.0,148.5106


In [121]:
sheet1.loc[(sheet1["order_id"].isna()) & (sheet1["customer_name"] == "Nick Radford"),"order_id"] = "CA-2020-152688"

In [122]:
sheet1[sheet1.isna().any(axis=1)]

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country_region,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
2251,2235.0,CA-2021-104066,2021-12-05,2021-12-10,Standard Class,QJ-19255,Quincy Jones,Corporate,United States,Burlington,...,NaN,East,TEC-AC-10001013,Technology,Accessories,Logitech ClearChat Comfort/USB Headset H390,205.03,7,0.0,67.6599
5291,5275.0,CA-2019-162887,2019-11-07,2019-11-09,Second Class,SV-20785,Stewart Visinsky,Consumer,United States,Burlington,...,NaN,East,FUR-CH-10000595,Furniture,Chairs,Safco Contoured Stacking Chairs,715.20,3,0.0,178.8000
8815,8799.0,US-2020-150140,2020-04-06,2020-04-10,Standard Class,VM-21685,Valerie Mitchum,Home Office,United States,Burlington,...,NaN,East,TEC-PH-10002555,Technology,Phones,Nortel Meridian M5316 Digital phone,1294.75,5,0.0,336.6350
9163,9147.0,US-2020-165505,2020-01-23,2020-01-27,Standard Class,CB-12535,Claudia Bergmann,Corporate,United States,Burlington,...,NaN,East,TEC-AC-10002926,Technology,Accessories,Logitech Wireless Marathon Mouse M705,99.98,2,0.0,42.9914
9164,9148.0,US-2020-165505,2020-01-23,2020-01-27,Standard Class,CB-12535,Claudia Bergmann,Corporate,United States,Burlington,...,NaN,East,OFF-AR-10003477,Office Supplies,Art,4009 Highlighters,8.04,6,0.0,2.7336
9165,9149.0,US-2020-165505,2020-01-23,2020-01-27,Standard Class,CB-12535,Claudia Bergmann,Corporate,United States,Burlington,...,NaN,East,OFF-ST-10001526,Office Supplies,Storage,Iceberg Mobile Mega Data/Printer Cart,1564.29,13,0.0,406.7154
9403,9387.0,US-2021-127292,2021-01-19,2021-01-23,Standard Class,RM-19375,Raymond Messe,Consumer,United States,Burlington,...,NaN,East,OFF-PA-10000157,Office Supplies,Paper,Xerox 191,79.92,4,0.0,37.5624
9404,9388.0,US-2021-127292,2021-01-19,2021-01-23,Standard Class,RM-19375,Raymond Messe,Consumer,United States,Burlington,...,NaN,East,OFF-PA-10001970,Office Supplies,Paper,Xerox 1881,12.28,1,0.0,5.7716
9405,9389.0,US-2021-127292,2021-01-19,2021-01-23,Standard Class,RM-19375,Raymond Messe,Consumer,United States,Burlington,...,NaN,East,OFF-AP-10000828,Office Supplies,Appliances,Avanti 4.4 Cu. Ft. Refrigerator,542.94,3,0.0,152.0232
9406,9390.0,US-2021-127292,2021-01-19,2021-01-23,Standard Class,RM-19375,Raymond Messe,Consumer,United States,Burlington,...,NaN,East,OFF-EN-10001509,Office Supplies,Envelopes,Poly String Tie Envelopes,2.04,1,0.0,0.9588


In [123]:
filtered_df = sheet1[(sheet1["city"] == "Burlington")]
filtered_df

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country_region,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
687,684.0,US-2021-168116,2021-11-04,2021-11-04,Same Day,GT-14635,Grant Thornton,Corporate,United States,Burlington,...,27217.0,South,TEC-MA-10004125,Technology,Machines,Cubify CubeX 3D Printer Triple Head Print,7999.980,4,0.5,-3839.9904
688,685.0,US-2021-168116,2021-11-04,2021-11-04,Same Day,GT-14635,Grant Thornton,Corporate,United States,Burlington,...,27217.0,South,OFF-AP-10002457,Office Supplies,Appliances,Eureka The Boss Plus 12-Amp Hard Box Upright V...,167.440,2,0.2,14.6510
1012,1009.0,US-2021-106705,2021-12-26,2022-01-01,Standard Class,PO-18850,Patrick O'Brill,Consumer,United States,Burlington,...,52601.0,Central,OFF-PA-10001509,Office Supplies,Paper,"Recycled Desk Saver Line ""While You Were Out"" ...",44.750,5,0.0,20.5850
1042,1039.0,CA-2021-121818,2021-11-20,2021-11-21,First Class,JH-15430,Jennifer Halladay,Consumer,United States,Burlington,...,27217.0,South,OFF-AR-10000203,Office Supplies,Art,Newell 336,23.968,7,0.2,2.6964
1043,1040.0,CA-2021-121818,2021-11-20,2021-11-21,First Class,JH-15430,Jennifer Halladay,Consumer,United States,Burlington,...,27217.0,South,OFF-AR-10004790,Office Supplies,Art,Staples in misc. colors,28.728,3,0.2,1.7955
1397,1394.0,CA-2021-124828,2021-07-03,2021-07-04,First Class,YS-21880,Yana Sorensen,Corporate,United States,Burlington,...,27217.0,South,OFF-AR-10003514,Office Supplies,Art,4009 Highlighters by Sanford,9.552,3,0.2,1.5522
2251,2235.0,CA-2021-104066,2021-12-05,2021-12-10,Standard Class,QJ-19255,Quincy Jones,Corporate,United States,Burlington,...,NaN,East,TEC-AC-10001013,Technology,Accessories,Logitech ClearChat Comfort/USB Headset H390,205.030,7,0.0,67.6599
2945,2929.0,US-2021-120390,2021-10-19,2021-10-26,Standard Class,TH-21550,Tracy Hopkins,Home Office,United States,Burlington,...,27217.0,South,OFF-BI-10004995,Office Supplies,Binders,GBC DocuBind P400 Electric Binding System,1633.188,4,0.7,-1306.5504
5082,5066.0,CA-2021-142090,2021-11-30,2021-12-07,Standard Class,SC-20380,Shahid Collister,Consumer,United States,Burlington,...,27217.0,South,TEC-AC-10002001,Technology,Accessories,Logitech Wireless Gaming Headset G930,383.976,3,0.2,81.5949
5083,5067.0,CA-2021-142090,2021-11-30,2021-12-07,Standard Class,SC-20380,Shahid Collister,Consumer,United States,Burlington,...,27217.0,South,FUR-TA-10001889,Furniture,Tables,Bush Advantage Collection Racetrack Conference...,1781.682,7,0.4,-653.2834


In [124]:
#Setting missing postal codes with a place holder value for all burlington city

sheet1.loc[(sheet1["city"].str.lower() == "burlington") & (sheet1["state"].str.lower() == "vermont") & (sheet1["postal_code"].isna()),"postal_code"] = 27217.0

In [125]:
sheet1.isnull().sum()

row_id            0
order_id          0
order_date        0
ship_date         0
ship_mode         0
customer_id       0
customer_name     0
segment           0
country_region    0
city              0
state             0
postal_code       0
region            0
product_id        0
category          0
sub_category      0
product_name      0
sales             0
quantity          0
discount          0
profit            0
dtype: int64

In [126]:
sheet1.shape

(9997, 21)

# Making the enitity dataframes

In [127]:
print(sheet1.columns)
print(sheet2.columns)
print(sheet3.columns)

Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'segment', 'country_region', 'city',
       'state', 'postal_code', 'region', 'product_id', 'category',
       'sub_category', 'product_name', 'sales', 'quantity', 'discount',
       'profit'],
      dtype='object')
Index(['region', 'first_name', 'last_name'], dtype='object')
Index(['returned', 'order_id'], dtype='object')


In [128]:
# from sheet1:
orders = sheet1[["order_id", "order_date", "customer_id"]]
order_details = sheet1[["order_id", "order_date", "quantity","product_id"]]
products = sheet1[["product_id", "order_id", "category", "sub_category", "product_name"]]

customers = sheet1[["customer_id", "customer_name", "segment"]]
addresses = sheet1[["country_region", "city", "state", "postal_code"]]
revenue = sheet1[["sales", "quantity", "discount", "profit"]] 
shipment = sheet1[["ship_date", "ship_mode"]]

# from sheet 2: 
people = sheet2[["region", "first_name", "last_name"]]
regions = sheet2[["region"]]

# from sheet 3:
returns = sheet3[["returned", "order_id"]]

#Adding id columns
regions.insert(0, "region_id", range(1, len(sheet2) + 1))
addresses.insert(0, "address_id", range(1, len(sheet1) + 1))
shipment.insert(0, "shipment_id", range(1, len(sheet1) + 1))
people.insert(0, "manager_id", range(1, len(sheet2) + 1))


# Converting the dataframes into a csv file

In [129]:
with pd.ExcelWriter("E2_GBP_DataSource_aligned.xlsx", engine="openpyxl") as writer:
    orders.to_excel(writer, sheet_name="orders", index=False)
    order_details.to_excel(writer, sheet_name="order_details", index=False)
    people.to_excel(writer, sheet_name="People", index=False)
    returns.to_excel(writer, sheet_name="Returns", index=False)
    products.to_excel(writer, sheet_name="products", index=False)
    customers.to_excel(writer, sheet_name="customers", index=False)
    addresses.to_excel(writer, sheet_name="addresses", index=False)
    regions.to_excel(writer, sheet_name="regions", index=False)
    revenue.to_excel(writer, sheet_name="revenue", index=False)
    shipment.to_excel(writer, sheet_name="shipment", index=False)
# zipcodes.csv is already a csv file